# SleepGuard: LOPO evaluation

This notebook implements held-out predictions, pooled AUROC with 1,000 patient-cluster
bootstrap repetitions, and per-patient AUROC. The configuration matches the **99-row,
16-column model_data.parquet schema shown in your October 26 screenshot**.
This notebook was run on all 99 records in the supplied JSON-lines export
(`Pasted text.txt`), containing 10 participants and 31 low nights. Its numeric
night values were decoded as Unix milliseconds before reconstructing a local
Parquet input. The original Parquet binary was not supplied. Saved outputs below
are the resulting first held-out predictions and evaluation results.

**Preliminary quality limitation:** 20 of the 99 nights have every HRV window
marked as needing review. They remain in this first run because the supplied
`Model_Eligible` flag is True for every night. Review those flags and verify
pre-sleep timing upstream before treating these results as validated evidence.

**Models:** regularized logistic regression (primary) and a shallow random forest
(comparison). Proposed feature variants: A = pre-sleep glucose mean and slope;
B1 = A + RMSSD; B2 = A + LF/HF; B3 = A + both. Confirm these variant definitions
with the team before interpreting results.

Save as `notebooks/03_lopo_evaluation.ipynb` in SleepGuard. Select your Python kernel,
restart it, and run all cells. If packages are missing, run
`%pip install numpy pandas scikit-learn pyarrow` once, then restart the kernel.
The input file is read only. Each completed run saves a new results folder.

## 0. Input and feature definitions

Input: `data/physiocgm/processed/model_data.parquet` inside your repo.
The supplied screenshot shows these actual columns:

| Actual column | Internal name / role |
| --- | --- |
| participant_id | subject: patient grouping for LOPO |
| night | night_id: patient-night identifier |
| hypo | hypo_night: outcome, 0 or 1 |
| mean_glucose_presleep | glucose predictor |
| glucose_slope_presleep | glucose predictor |
| rmssd_presleep | HRV predictor |
| lf_hf_presleep | lfhf_presleep: HRV predictor |
| Model_Eligible | upstream eligibility flag; not a predictor |

The old PDF's 1/2/3-hour glucose columns are absent from this merged table; this
version uses its two available pre-sleep glucose features. No values are invented.
The other coverage, quality, and outcome-related columns are not model inputs.

`night` is used only to identify a night. It is **not treated as actual sleep onset
or a prediction timestamp**. Upstream extraction must document the prediction time,
timezone, pre-sleep window lengths, and ensure features end before prediction.
If using 22:00 as a proxy, describe it as a clock-time cutoff. Overnight HRV must
not be substituted for pre-sleep HRV.

Only rows with `Model_Eligible == True` are considered. Missing eligibility flags
are excluded. The notebook trusts this upstream flag; it cannot independently
verify the team's coverage, episode-label, or already-low-at-prediction rules.
The default complete-case analysis then excludes rows missing any of the four
predictors. All algorithms and variants use the same remaining nights.
The retained cohort may therefore contain fewer than the 99 input rows.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score

In [2]:
DOWNLOADS = Path.home() / "Downloads"
# Finds the repo when the kernel starts in the root or notebooks/ directory.
# If your kernel starts elsewhere, set REPO_ROOT to the repo's absolute path.
def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "physiocgm" / "processed" / "model_data.parquet").is_file():
            return candidate
    return start.parent if start.name == "notebooks" else start

REPO_ROOT = find_repo_root(Path.cwd())
INPUT_FILE = REPO_ROOT / "data" / "physiocgm" / "processed" / "model_data.parquet"
OUTPUT_ROOT = REPO_ROOT / "results" / "lopo"

# Names observed in your actual file's schema screenshot.
# night_id is a row identifier, not an assumed bedtime or prediction timestamp.
COLUMN_RENAME = {
    "participant_id": "subject",
    "night": "night_id",
    "hypo": "hypo_night",
    "lf_hf_presleep": "lfhf_presleep",
}
ELIGIBILITY_COLUMN = "Model_Eligible"
GLUCOSE_FEATURES = ["mean_glucose_presleep", "glucose_slope_presleep"]

# PROPOSED B1/B2/B3 mapping; replace if the team's agreed mapping differs.
FEATURE_SETS = {
    "A": GLUCOSE_FEATURES,
    "B1": GLUCOSE_FEATURES + ["rmssd_presleep"],
    "B2": GLUCOSE_FEATURES + ["lfhf_presleep"],
    "B3": GLUCOSE_FEATURES + ["rmssd_presleep", "lfhf_presleep"],
}

# PROPOSED algorithms. Fixed starting settings, not tuned on held-out patients.
# Add/remove algorithms here to match the agreed list. Each must implement
# predict_proba; arbitrary SVM margins should not be silently pooled as probabilities.
CLASSIFIERS = {
    "logistic": LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000),
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=3, min_samples_leaf=10,
        random_state=42, n_jobs=-1,
    ),
}

# Both options use exactly the same evaluation rows for every feature set.
# complete_case: exclude a night if any required feature is missing (default).
# median_impute: retain partially observed nights and impute using training patients.
# Entirely unobserved glucose or HRV must already fail your upstream quality criteria.
MISSING_POLICY = "complete_case"
N_BOOTSTRAPS = 1000
BOOTSTRAP_SEED = 20261026

## Inspect the input

Print the file path, shape, columns, and data types before modeling. The column
mapping above is based on your screenshot; this cell reads the actual file.
The following section reports eligibility exclusions and the final patient/night
counts. The screenshot's 99 rows are input rows, not an assumed final cohort.

In [3]:
def read_model_table(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Real model table not found: {path}. Put model_data.parquet in "
            "data/physiocgm/processed/ inside your repo, or set INPUT_FILE explicitly."
        )
    suffix = path.suffix.lower()
    if suffix in (".parquet", ".pq"):
        try:
            return pd.read_parquet(path)
        except ImportError as exc:
            raise ImportError("Install the Parquet reader in this notebook's kernel: %pip install pyarrow") from exc
    if suffix == ".csv":
        return pd.read_csv(path, dtype={"subject": str})
    raise ValueError("INPUT_FILE must be a .parquet, .pq, or .csv file.")

raw_preview = read_model_table(INPUT_FILE)
print("Input file:", INPUT_FILE)
print("Rows and columns:", raw_preview.shape)
print("Columns:", raw_preview.columns.tolist())
print("Index names:", raw_preview.index.names)
print(raw_preview.dtypes.to_string())

Input file: /Users/aryan/Documents/GitHub/SleepGuard/data/physiocgm/processed/model_data.parquet
Rows and columns: (99, 16)
Columns: ['participant_id', 'night', 'mean_glucose_presleep', 'glucose_slope_presleep', 'rmssd_presleep', 'lf_hf_presleep', 'n_hrv_windows', 'hrv_missing', 'hypo', 'n_glucose_readings', 'n_hrv_windows_needing_review', 'outcome_coverage', 'baseline_known', 'low_at_prediction', 'outcome_complete', 'Model_Eligible']
Index names: [None]
participant_id                          object
night                           datetime64[ns]
mean_glucose_presleep                  float64
glucose_slope_presleep                 float64
rmssd_presleep                         float64
lf_hf_presleep                         float64
n_hrv_windows                            int64
hrv_missing                              int64
hypo                                     Int64
n_glucose_readings                       int64
n_hrv_windows_needing_review             int64
outcome_coverage        

## 1. Validate the table and finish LOPO

Each patient is held out once. All of their nights stay together. Imputation,
scaling, and classifier fitting use only the other patients. This first run uses
fixed settings. If you tune later, use patient-grouped inner cross-validation
inside each outer training fold; never choose settings using held-out outcomes.

Default complete-case analysis may reduce the cohort. The exclusion report and
before/after patient counts show exactly what changed. Do not compare baseline A
on all nights with B on a smaller cohort and call the difference an HRV effect.
Rows failing the upstream eligibility flag are excluded before fitting.
Exclusion reasons and per-patient counts are saved with the results.

In [4]:
def prepare_cohort(raw, feature_sets, missing_policy):
    if set(feature_sets) != {"A", "B1", "B2", "B3"}:
        raise ValueError("Define exactly A, B1, B2, and B3 in FEATURE_SETS.")
    features = list(dict.fromkeys(c for cols in feature_sets.values() for c in cols))
    prohibited = {"subject", "night_id", "hypo_night", "hypo_events",
                  "mean_glucose", "min_glucose", "cgm_hours", ELIGIBILITY_COLUMN,
                  "outcome_coverage", "outcome_complete", "low_at_prediction"}
    if prohibited.intersection(features):
        raise ValueError(f"Identifiers, labels, eligibility, or outcome data in predictors: {prohibited.intersection(features)}")
    if any(not cols or len(cols) != len(set(cols)) for cols in feature_sets.values()):
        raise ValueError("Each feature set must be nonempty and contain no duplicate columns.")
    keys = ["subject", "night_id", "hypo_night"]
    required = keys + features + [ELIGIBILITY_COLUMN]
    missing = sorted(set(required) - set(raw.columns))
    if missing:
        raise ValueError(f"Missing input columns: {missing}. Check COLUMN_RENAME and FEATURE_SETS.")
    df = raw[required].copy().reset_index(drop=True)
    if df[["subject", "night_id"]].isna().any().any():
        raise ValueError("Missing patient ID or night identifier.")
    df["subject"] = df["subject"].astype(str).str.strip()
    if df["subject"].eq("").any():
        raise ValueError("Empty patient ID.")
    df["night_id"] = pd.to_datetime(df["night_id"], errors="raise")
    if df["night_id"].isna().any():
        raise ValueError("Invalid night identifier.")
    if df.duplicated(["subject", "night_id"]).any():
        raise ValueError("Duplicate patient-night rows. Aggregate HRV before merging.")

    flag = df[ELIGIBILITY_COLUMN]
    if not flag.dropna().isin([True, False]).all():
        raise ValueError(f"{ELIGIBILITY_COLUMN} must contain booleans or 0/1, not strings.")
    eligible = flag.eq(True).fillna(False).astype(bool)
    df["hypo_night"] = pd.to_numeric(df["hypo_night"], errors="raise")
    if not df["hypo_night"].dropna().isin([0, 1]).all():
        raise ValueError("Known hypo labels must be 0 or 1.")
    if df.loc[eligible, "hypo_night"].isna().any():
        raise ValueError("An eligible night has no outcome label; correct upstream eligibility/labeling.")
    for col in features:
        df[col] = pd.to_numeric(df[col], errors="raise")
    df[features] = df[features].replace([np.inf, -np.inf], np.nan)
    if missing_policy == "complete_case":
        feature_keep = df[features].notna().all(axis=1)
    elif missing_policy == "median_impute":
        feature_keep = pd.Series(True, index=df.index)
        for cols in feature_sets.values():
            feature_keep &= df[cols].notna().any(axis=1)
    else:
        raise ValueError("MISSING_POLICY must be complete_case or median_impute.")
    keep = eligible & feature_keep

    excluded = df.loc[~keep, keys + [ELIGIBILITY_COLUMN]].copy()
    reasons = pd.Series("missing_predictors", index=df.index)
    reasons.loc[~eligible] = "model_eligible_false"
    reasons.loc[flag.isna()] = "model_eligible_missing"
    excluded["exclusion_reason"] = reasons.loc[~keep]
    excluded["missing_features"] = [
        ",".join(row.index[row])
        for _, row in df.loc[~keep, features].isna().iterrows()
    ]
    counts = df.groupby("subject")["hypo_night"].agg(
        input_nights="size", input_labeled_nights="count", input_low_nights="sum")
    eligible_counts = df.loc[eligible].groupby("subject").size().rename("model_eligible_nights")
    cohort = df.loc[keep, keys + features].copy()
    cohort["hypo_night"] = cohort["hypo_night"].astype(int)
    cohort = cohort.sort_values(["subject", "night_id"]).reset_index(drop=True)
    retained = cohort.groupby("subject")["hypo_night"].agg(
        included_nights="size", included_low_nights="sum")
    counts = counts.join(eligible_counts).join(retained).fillna(0).astype(int).reset_index()
    counts["eligibility_excluded_nights"] = counts["input_nights"] - counts["model_eligible_nights"]
    counts["missing_predictor_excluded_nights"] = counts["model_eligible_nights"] - counts["included_nights"]
    counts["included_nonlow_nights"] = counts["included_nights"] - counts["included_low_nights"]
    print(f"Input: {len(df)} nights; upstream eligible: {int(eligible.sum())}; "
          f"eligible but missing required predictors: {int((eligible & ~feature_keep).sum())}.")
    if cohort["subject"].nunique() < 3 or cohort["hypo_night"].nunique() < 2:
        raise ValueError("Insufficient eligible patients or outcome classes after filtering; inspect Model_Eligible and missing features.")
    return cohort, excluded.reset_index(drop=True), counts


def generate_oof(df, feature_sets, classifiers):
    if not classifiers:
        raise ValueError("No classifiers configured.")
    if any("__" in name for name in classifiers):
        raise ValueError("Classifier names cannot contain '__'.")
    y = df["hypo_night"].to_numpy()
    groups = df["subject"].to_numpy()
    oof = df[["subject", "night_id", "hypo_night"]].copy()
    score_columns = [(name, variant, f"{name}__{variant}")
                     for name in classifiers for variant in feature_sets]
    for _, _, col in score_columns:
        oof[col] = np.nan
    counts = np.zeros(len(df), dtype=int)
    audit = []
    for train_idx, test_idx in LeaveOneGroupOut().split(df, y, groups):
        held_out = str(groups[test_idx][0])
        train_subjects = sorted(set(groups[train_idx]))
        assert held_out not in train_subjects
        if np.unique(y[train_idx]).size != 2:
            raise ValueError(f"Training data for held-out {held_out} has only one class; cannot fit agreed classifiers.")
        for name, variant, col in score_columns:
            features = feature_sets[variant]
            train_X = df.iloc[train_idx][features]
            if train_X.isna().all().any():
                bad = train_X.columns[train_X.isna().all()].tolist()
                raise ValueError(f"{held_out}/{variant}: all-missing training features {bad}.")
            model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), clone(classifiers[name]))
            model.fit(train_X, y[train_idx])
            class_index = int(np.flatnonzero(model.classes_ == 1)[0])
            predictions = model.predict_proba(df.iloc[test_idx][features])[:, class_index]
            oof.loc[test_idx, col] = predictions
        counts[test_idx] += 1
        audit.append({"held_out_subject": held_out,
                      "train_subjects": ",".join(train_subjects),
                      "train_nights": len(train_idx), "test_nights": len(test_idx),
                      "train_low_nights": int(y[train_idx].sum()),
                      "test_low_nights": int(y[test_idx].sum())})
        print(f"Held out {held_out}: trained on {len(train_subjects)} patients; predicted {len(test_idx)} nights.")
    assert np.all(counts == 1), "Every night must be predicted exactly once."
    scores = oof[[c for _, _, c in score_columns]].to_numpy()
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
    return oof, pd.DataFrame(audit), score_columns

In [5]:
# Reload so edits to COLUMN_RENAME take effect without stale data.
raw = read_model_table(INPUT_FILE).rename(columns=COLUMN_RENAME)
if raw.columns.duplicated().any():
    raise ValueError("COLUMN_RENAME created duplicate column names.")
cohort, excluded_nights, cohort_counts = prepare_cohort(raw, FEATURE_SETS, MISSING_POLICY)
print(cohort_counts.to_string(index=False))
print(f"Included {len(cohort)} nights from {cohort.subject.nunique()} patients; "
      f"{int(cohort.hypo_night.sum())} low nights; excluded {len(excluded_nights)} nights.")
oof, fold_audit, score_columns = generate_oof(cohort, FEATURE_SETS, CLASSIFIERS)

Input: 99 nights; upstream eligible: 99; eligible but missing required predictors: 0.
subject  input_nights  input_labeled_nights  input_low_nights  model_eligible_nights  included_nights  included_low_nights  eligibility_excluded_nights  missing_predictor_excluded_nights  included_nonlow_nights
  c1s01             7                     7                 3                      7                7                    3                            0                                  0                       4
  c1s02            11                    11                 0                     11               11                    0                            0                                  0                      11
  c1s03            10                    10                 2                     10               10                    2                            0                                  0                       8
  c1s04             6                     6                 1         

## 2. Pooled AUROC and paired patient bootstrap

Calculate pooled AUROC once on the concatenated held-out scores. Then draw
the same number of patient IDs as in the eligible cohort, with replacement,
including all nights each time a patient is drawn. A repeated patient contributes
their nights repeatedly. Use identical resamples for all model variants.
Skip replicates with one label class and report how many were skipped.

The 2.5th and 97.5th percentiles of approximately 1,000 valid replicates form an
approximate 95% interval. We also compute paired differences B1-A, B2-A, and B3-A.
Prespecify the primary comparison (suggested: logistic B3-A); the other comparisons
are exploratory. Separate model intervals do not replace an interval for the difference.

These are patient-cluster percentile intervals conditional on the saved OOF scores.
They do not fully capture refitting, model-selection uncertainty, or dependence
created by overlapping training folds. With only about 10 patients they remain
exploratory. Pooled AUROC includes cross-patient comparisons; it is not the average
patient AUROC or a guarantee of performance on a new population.

In [6]:
def paired_patient_bootstrap(oof, score_columns, n_bootstraps=1000, seed=42):
    if n_bootstraps < 1:
        raise ValueError("n_bootstraps must be positive.")
    y = oof["hypo_night"].to_numpy()
    scores = oof[[c for _, _, c in score_columns]].to_numpy()
    subjects = oof["subject"].to_numpy()
    unique = np.unique(subjects)
    indices = {p: np.flatnonzero(subjects == p) for p in unique}
    point = np.array([roc_auc_score(y, scores[:, j]) for j in range(scores.shape[1])])
    rng = np.random.default_rng(seed)
    draws, rows, skipped, attempts = [], [], 0, 0
    while len(rows) < n_bootstraps and attempts < 20 * n_bootstraps:
        attempts += 1
        sample = rng.choice(unique, size=len(unique), replace=True)
        # Concatenation preserves multiplicity; isin(sample) would incorrectly drop duplicates.
        idx = np.concatenate([indices[p] for p in sample])
        if np.unique(y[idx]).size < 2:
            skipped += 1
            continue
        rows.append([roc_auc_score(y[idx], scores[idx, j]) for j in range(scores.shape[1])])
        draws.append({"replicate": len(rows), "sampled_subjects": ",".join(sample),
                      "sampled_nights": len(idx)})
    if len(rows) != n_bootstraps:
        raise ValueError(f"Only {len(rows)} valid bootstrap replicates in {attempts} attempts.")
    boot = np.array(rows)
    bounds = np.quantile(boot, [0.025, 0.975], axis=0)
    summary = []
    delta = []
    position = {(name, variant): j for j, (name, variant, _) in enumerate(score_columns)}
    for j, (name, variant, _) in enumerate(score_columns):
        summary.append({"algorithm": name, "feature_set": variant,
                        "n_patients": len(unique), "n_nights": len(y), "n_low_nights": int(y.sum()),
                        "pooled_auroc": point[j], "ci_lower": bounds[0, j], "ci_upper": bounds[1, j],
                        "valid_bootstraps": len(rows), "skipped_single_class": skipped})
        if variant != "A":
            a = position[(name, "A")]
            delta_bounds = np.quantile(boot[:, j] - boot[:, a], [0.025, 0.975])
            delta.append({"algorithm": name, "comparison": f"{variant}-A",
                          "delta_auroc": point[j] - point[a],
                          "ci_lower": delta_bounds[0], "ci_upper": delta_bounds[1],
                          "valid_bootstraps": len(rows)})
    bootstrap_values = pd.DataFrame(boot, columns=[c for _, _, c in score_columns])
    bootstrap_values.insert(0, "replicate", np.arange(1, len(rows) + 1))
    return pd.DataFrame(summary), pd.DataFrame(delta), bootstrap_values, pd.DataFrame(draws)

In [7]:
print(f"Computing {N_BOOTSTRAPS:,} paired patient bootstrap repetitions...")
pooled_results, delta_results, bootstrap_values, bootstrap_draws = paired_patient_bootstrap(
    oof, score_columns, N_BOOTSTRAPS, BOOTSTRAP_SEED
)
print(pooled_results.round(4).to_string(index=False))
print("\nPaired improvement over A:")
print(delta_results.round(4).to_string(index=False))

Computing 1,000 paired patient bootstrap repetitions...
    algorithm feature_set  n_patients  n_nights  n_low_nights  pooled_auroc  ci_lower  ci_upper  valid_bootstraps  skipped_single_class
     logistic           A          10        99            31        0.5721    0.4417    0.6935              1000                     0
     logistic          B1          10        99            31        0.5655    0.4418    0.6795              1000                     0
     logistic          B2          10        99            31        0.5280    0.4176    0.6355              1000                     0
     logistic          B3          10        99            31        0.5408    0.4130    0.6705              1000                     0
random_forest           A          10        99            31        0.5745    0.4296    0.7004              1000                     0
random_forest          B1          10        99            31        0.5598    0.4484    0.6830              1000               

## 3. Per-held-out-patient AUROC

Compute each patient's AUROC using only their saved held-out predictions.
If they have zero low nights OR zero non-low nights, record N/A with the reason.
Do not remove these patients from the pooled calculation.

The PDF's displayed results give c1s02 and c1s04 one low night and eleven non-low
nights each. Both classes are present, so their AUROCs are defined; the estimates
are simply very unstable. Recheck these counts in the final eligible cohort.

In [8]:
def per_patient_auroc(oof, score_columns):
    rows = []
    names = list(dict.fromkeys(name for name, _, _ in score_columns))
    for subject, patient in oof.groupby("subject", sort=True):
        y = patient["hypo_night"].to_numpy()
        low = int(y.sum())
        normal = len(y) - low
        valid = low > 0 and normal > 0
        for name in names:
            row = {"subject": subject, "algorithm": name, "n_nights": len(y),
                   "n_low_nights": low, "n_nonlow_nights": normal,
                   "status": "ok" if valid else "undefined_single_class"}
            for model_name, variant, col in score_columns:
                if model_name == name:
                    row[f"auroc_{variant}"] = roc_auc_score(y, patient[col]) if valid else np.nan
            for variant in ("B1", "B2", "B3"):
                row[f"delta_{variant}_minus_A"] = row[f"auroc_{variant}"] - row["auroc_A"]
            rows.append(row)
    return pd.DataFrame(rows)

patient_results = per_patient_auroc(oof, score_columns)
print(patient_results.round(4).to_string(index=False, na_rep="N/A"))

# Descriptive macro-average: each evaluable patient has equal weight.
# Report the denominator; this differs from pooled AUROC.
macro_rows = []
for name, group in patient_results.groupby("algorithm", sort=False):
    valid = group.loc[group.status == "ok"]
    row = {"algorithm": name, "evaluable_patients": len(valid), "total_patients": len(group)}
    row.update({f"mean_patient_auroc_{v}": valid[f"auroc_{v}"].mean() for v in FEATURE_SETS})
    macro_rows.append(row)
macro_results = pd.DataFrame(macro_rows)
print("\nDescriptive patient-average AUROCs:")
print(macro_results.round(4).to_string(index=False))

subject     algorithm  n_nights  n_low_nights  n_nonlow_nights                 status  auroc_A  auroc_B1  auroc_B2  auroc_B3  delta_B1_minus_A  delta_B2_minus_A  delta_B3_minus_A
  c1s01      logistic         7             3                4                     ok   0.3333    0.3333    0.3333    0.3333            0.0000            0.0000            0.0000
  c1s01 random_forest         7             3                4                     ok   0.2500    0.5000    0.3333    0.4167            0.2500            0.0833            0.1667
  c1s02      logistic        11             0               11 undefined_single_class      N/A       N/A       N/A       N/A               N/A               N/A               N/A
  c1s02 random_forest        11             0               11 undefined_single_class      N/A       N/A       N/A       N/A               N/A               N/A               N/A
  c1s03      logistic        10             2                8                     ok   0.5000    0.5000 

## Save the deliverables

Each run receives a new folder under `results/lopo/` inside your repo. Share the notebook and the three main result files:
`oof_predictions.csv`, `pooled_auroc_ci.csv`, and `patient_auroc.csv`.
Also share `paired_auc_differences.csv` to support the HRV comparison.
The remaining files document exclusions, folds, bootstrap draws, and configuration.

These are research evaluation outputs. An AUROC is a ranking measure, not an
accuracy percentage or evidence of calibrated individual risk. Alarm sensitivity,
specificity, and precision need a separately defined threshold chosen using training data.

In [9]:
run_id = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
out_dir = OUTPUT_ROOT / run_id
out_dir.mkdir(parents=True, exist_ok=False)
outputs = {
    "oof_predictions.csv": oof,
    "pooled_auroc_ci.csv": pooled_results,
    "patient_auroc.csv": patient_results,
    "paired_auc_differences.csv": delta_results,
    "patient_macro_auroc.csv": macro_results,
    "cohort_counts.csv": cohort_counts,
    "excluded_nights.csv": excluded_nights,
    "fold_audit.csv": fold_audit,
    "bootstrap_auroc.csv": bootstrap_values,
    "bootstrap_patient_draws.csv": bootstrap_draws,
}
for filename, frame in outputs.items():
    frame.to_csv(out_dir / filename, index=False, na_rep="NA")
metadata = {
    "input_file": str(INPUT_FILE.resolve()),
    "input_sha256": hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest(),
    "column_rename": COLUMN_RENAME,
    "eligibility_column": ELIGIBILITY_COLUMN,
    "eligibility_rule": "Upstream flag is True; missing flags excluded. Not independently audited.",
    "night_identifier": "Calendar night key; not an inferred prediction timestamp or sleep onset.",
    "feature_sets": FEATURE_SETS,
    "classifiers": {name: model.get_params() for name, model in CLASSIFIERS.items()},
    "missing_policy": MISSING_POLICY,
    "n_bootstraps": N_BOOTSTRAPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__},
    "interval_method": "paired patient-cluster percentile bootstrap of fixed LOPO predictions",
    "interpretation": "Exploratory; does not fully capture refitting/model-selection uncertainty.",
}
(out_dir / "run_config.json").write_text(json.dumps(metadata, indent=2, default=str))
print("Saved results to:", out_dir)

Saved results to: /Users/aryan/Documents/GitHub/SleepGuard/results/lopo/run_20261027T023451_454319Z


## References

- [LeaveOneGroupOut](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html)
- [Preventing data leakage with pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- [Nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [Cross-validated prediction metric caveats](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)
- [Cluster bootstrap for correlated ROC data](https://www.frontiersin.org/journals/veterinary-science/articles/10.3389/fvets.2023.1274786/full)

Source inspection: supplied `Event Count (1).pdf`; repository branches `aryan-01`
and `adi-01` inspected October 26, 2026. No existing LOPO loop was found in
`01_ecg_rpeaks.ipynb`, `data_collection.ipynb`, or `hrv_features.ipynb`.
Schema update: supplied October 26 screenshots show participant_id, night, hypo,
mean_glucose_presleep, glucose_slope_presleep, rmssd_presleep, lf_hf_presleep,
and Model_Eligible. The subsequent supplied JSON-lines export was used for
the 99-night execution shown here; no extra quality exclusions or model tuning
were introduced after inspecting these results.